In [13]:
import pandas as pd
import json

# Load the un-annotated dataset
df = pd.read_csv('../final_dataset.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Names:\n{list(df.columns)}")
print(f"\nFirst few rows:")
display(df.head())

print(f"\n--- Basic Statistics ---")
print(f"Total rows: {len(df)}")
print(f"Unique fact_ids: {df['fact_id'].nunique()}")

# Check case_name structure (it's likely JSON)
print(f"\nFirst case_name example: {df['case_name'].iloc[0]}")

# Check unique cases
try:
    df['clean_case_name'] = df['case_name'].apply(
        lambda x: json.loads(x)[0] if isinstance(x, str) and x.startswith('[') else x
    )
    print(f"Unique cases: {df['clean_case_name'].nunique()}")
except:
    print("Could not parse case_name as JSON")

print(f"\n--- Label Distribution ---")
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True))

print(f"\n--- Type Distribution ---")
print(df['type'].value_counts())

print(f"\n--- Data Types ---")
print(df.dtypes)

print(f"\n--- Missing Values ---")
print(df.isnull().sum())

Dataset Shape: (12310, 10)

Column Names:
['claim', 'case_name', 'judgement', 'explanation', 'raw_output', 'type', 'overruling_case', 'fact_id', 'label', 'is_negation']

First few rows:


,claim,case_name,judgement,explanation,raw_output,type,overruling_case,fact_id,label,is_negation
0,The Constitution does not inherently recognize...,"[""Roe v. Wade""]",consistent,The case evidence explicitly states that the C...,"```json\n{\n ""explanation"": ""The case evide...",none,NaN,0,0,True
1,The Constitution protects a person's right to ...,"[""Roe v. Wade""]",consistent,The case evidence explicitly states that the C...,"```json\n{\n ""explanation"": ""The case evide...",none,NaN,0,1,False
2,States may regulate abortion procedures during...,"[""Roe v. Wade""]",consistent,"The case evidence explicitly states that ""In t...","```json\n{\n ""explanation"": ""The case evide...",none,NaN,1,0,True
3,States cannot restrict abortion during the fir...,"[""Roe v. Wade""]",consistent,"The case evidence explicitly states that ""In t...","```json\n{\n ""explanation"": ""The case evide...",none,NaN,1,1,False
4,"During the second three months of pregnancy, s...","[""Roe v. Wade""]",consistent,The case evidence states that in the second tr...,"```json\n{\n ""explanation"": ""The case evide...",none,NaN,2,1,False



--- Basic Statistics ---
Total rows: 12310
Unique fact_ids: 6401

First case_name example: ["Roe v. Wade"]
Unique cases: 2915

--- Label Distribution ---
label
1    6401
0    5909
Name: count, dtype: int64
label
1    0.519984
0    0.480016
Name: proportion, dtype: float64

--- Type Distribution ---
type
none         11378
overruled      492
confirmed      440
Name: count, dtype: int64

--- Data Types ---
claim              object
case_name          object
judgement          object
explanation        object
raw_output         object
type               object
overruling_case    object
fact_id             int64
label               int64
is_negation          bool
clean_case_name    object
dtype: object

--- Missing Values ---
claim                  0
case_name              0
judgement            492
explanation          492
raw_output           496
type                   0
overruling_case    11818
fact_id                0
label                  0
is_negation            0
clean_case_name  

In [23]:
# Load the human-annotated subset
df_anno = pd.read_excel('../supreme_court_test_claims.xlsx')

# --- Fix Fact IDs for Swapped Claims ---
try:
    # 1. Load original un-annotated set
    df_orig = pd.read_excel('sampled_claims_1000.xlsx')
    orig_claims_map = df_orig.set_index('fact_id')['claim'].to_dict()
    
    # 2. Load the diff case results to identify switched claims
    df_diff = pd.read_csv('../results/factual_claims_resolved_contradictions_diff_case.csv')
    # Collect all claims involved in diff case contradictions
    diff_claims_set = set(df_diff['claim1'].unique()) | set(df_diff['claim2'].unique())
    # Also add cleaned versions just in case
    diff_claims_set.update([str(c).replace('*', '').strip() for c in diff_claims_set])

    # 3. Prepare lookup for correct IDs from final dataset (df)
    # df is assumed to be loaded in the previous cell
    temp_df = df.copy()
    temp_df['clean_claim'] = temp_df['claim'].apply(lambda x: str(x).replace('*', '').strip())
    claim_to_id_map = temp_df.set_index('clean_claim')['fact_id'].to_dict()
    
    updated_count = 0
    
    for idx, row in df_anno.iterrows():
        fid = row['fact_id']
        current_claim = row['claim']
        clean_current = str(current_claim).replace('*', '').strip()
        
        # Check if we have the original claim for this ID
        if fid in orig_claims_map:
            orig_claim = orig_claims_map[fid]
            
            # If claim text is different
            if current_claim != orig_claim:
                # Check if this claim appears in the diff_case file (switched)
                if (current_claim in diff_claims_set) or (clean_current in diff_claims_set):
                    
                    # Find the correct ID in the final dataset
                    if clean_current in claim_to_id_map:
                        new_fid = claim_to_id_map[clean_current]
                        
                        # Update if the ID is different
                        if new_fid != fid:
                            df_anno.at[idx, 'fact_id'] = new_fid
                            updated_count += 1
    
    print(f"Updated {updated_count} fact_ids for switched claims.")

except Exception as e:
    print(f"Could not perform fact_id correction: {e}")

# --- Drop Duplicates ---
# Check for duplicate claims in the annotated set
duplicates = df_anno[df_anno.duplicated(subset=['claim'], keep=False)]
if not duplicates.empty:
    print(f"Found {len(duplicates)} duplicate claims in annotated set.")
    initial_len = len(df_anno)
    df_anno = df_anno.drop_duplicates(subset=['claim'], keep='first')
    print(f"Dropped {initial_len - len(df_anno)} duplicates. New size: {len(df_anno)}")
else:
    print("No duplicate claims found in annotated set.")
# -----------------------

print(f"Annotated Dataset Shape: {df_anno.shape}")
print(f"\nColumn Names:\n{list(df_anno.columns)}")
print(f"\nFirst few rows:")
display(df_anno.head())

print(f"\n--- Basic Statistics ---")
print(f"Total annotated rows: {len(df_anno)}")

# Check for annotation columns
annotation_cols = [col for col in df_anno.columns if col not in df.columns]
print(f"\nAnnotation-specific columns: {annotation_cols}")

# If there's a Rating or similar column, show distribution
if 'Rating' in df_anno.columns:
    print(f"\n--- Rating Distribution ---")
    print(df_anno['Rating'].value_counts())
    print(f"\nRatings (normalized):")
    print(df_anno['Rating'].value_counts(normalize=True))
    
    # Check how many are annotated vs not
    annotated_count = df_anno['Rating'].notna().sum()
    print(f"\nAnnotated: {annotated_count}")
    print(f"Not yet annotated: {len(df_anno) - annotated_count}")

# Check if fact_ids match the main dataset
if 'fact_id' in df_anno.columns:
    matching_ids = df_anno['fact_id'].isin(df['fact_id']).sum()
    print(f"\nFact IDs matching main dataset: {matching_ids}/{len(df_anno)}")

print(f"\n--- Label Distribution in Annotated Set ---")
if 'label' in df_anno.columns:
    print(df_anno['label'].value_counts())

print(f"\n--- Missing Values ---")
print(df_anno.isnull().sum())

Updated 7 fact_ids for switched claims.
Found 4 duplicate claims in annotated set.
Dropped 2 duplicates. New size: 998
Annotated Dataset Shape: (998, 12)

Column Names:
['claim', 'case_name', 'type', 'overruling_case', 'fact_id', 'label', 'facts', 'question', 'conclusion', 'class', 'Rating', 'old_negation']

First few rows:


,claim,case_name,type,overruling_case,fact_id,label,facts,question,conclusion,class,Rating,old_negation
0,Courts must evaluate whether mutual fund fees ...,"[""Jones v. Harris Associates L.P.""]",none,NaN,3628,1,Plaintiffs were investors in several mutual fu...,Did the Seventh Circuit err in holding that cl...,Yes. The Supreme Court held that the Seventh C...,Supported,1,NaN
1,Replacing county-wide elections with single-me...,"[""Rogers v. Lodge""]",none,NaN,742,1,"Eight black citizens of Burke County, Georgia,...",Did the system of elections violate the Fourte...,"In a 6-to-3 decision, the Court held that the ...",Supported,1,NaN
2,The Fifth Amendment takings clause does not cr...,"[""San Remo Hotel, L.P. v. City and County of S...",overruled,Stewart v. Smith,6184,2,The owners and operators of a hotel in San Fra...,Should federal courts make an exception to the...,No. In a 9-0 judgment delivered by Justice Joh...,Overruled,1,NaN
3,ERISA allows states to require employers to fu...,"[""UNUM Life Insurance Company of America v. Wa...",none,NaN,2219,0,UNUM Life Insurance Company of America (UNUM) ...,Does the Employee Retirement Income Security A...,"Yes and no. In a unanimous opinion, delivered ...",Refuted,1,ERISA explicitly permits states to require emp...
4,Police are legally prohibited from entering a ...,"[""Michigan v. Fisher""]",none,NaN,438,0,Jeremy Fisher was charged with assault with a ...,Did the Michigan Court of Appeals err when it ...,"Yes. In a per curiam opinion, the Supreme Cour...",Refuted,1,Police are legally prohibited from entering a ...



--- Basic Statistics ---
Total annotated rows: 998

Annotation-specific columns: ['facts', 'question', 'conclusion', 'class', 'Rating', 'old_negation']

--- Rating Distribution ---
Rating
1     572
0      64
1?      1
?       1
Name: count, dtype: int64

Ratings (normalized):
Rating
1     0.896552
0     0.100313
1?    0.001567
?     0.001567
Name: proportion, dtype: float64

Annotated: 638
Not yet annotated: 360

Fact IDs matching main dataset: 998/998

--- Label Distribution in Annotated Set ---
label
1    470
0    455
2     73
Name: count, dtype: int64

--- Missing Values ---
claim                0
case_name            0
type                 0
overruling_case    925
fact_id              0
label                0
facts                0
question             0
conclusion           0
class                0
Rating             360
old_negation       841
dtype: int64


In [15]:
import numpy as np

# Filter for annotated rows (where Rating is 1, excluding 0 which are bad claims)
annotated_mask = df_anno['Rating'] == 1
df_annotated = df_anno[annotated_mask].copy()

print(f"Total valid annotated rows (Rating=1): {len(df_annotated)}")
print("Labels found:", df_annotated['label'].unique())

# Separate Overruled (label=2) and others (0 and 1)
overruled_df = df_annotated[df_annotated['label'] == 2]
label_0_df = df_annotated[df_annotated['label'] == 0]
label_1_df = df_annotated[df_annotated['label'] == 1]

print(f"Overruled (2) count: {len(overruled_df)}")
print(f"Label 0 count: {len(label_0_df)}")
print(f"Label 1 count: {len(label_1_df)}")

# Target size
TARGET_SIZE = 500

# Start with all Overruled
final_df = overruled_df.copy()

# Calculate how many more we need
needed = TARGET_SIZE - len(final_df)

if needed > 0:
    # Balance Label 0 and Label 1
    # Add a small random jitter so they aren't exactly equal
    jitter = np.random.randint(-3, 4) # -3 to +3
    
    target_0 = (needed // 2) + jitter
    target_1 = needed - target_0
    
    # Ensure positive
    target_0 = max(0, target_0)
    target_1 = max(0, target_1)
    
    print(f"Targeting ~{target_0} of Label 0 and ~{target_1} of Label 1")

    # Helper to sample safely
    def sample_safe(df, n):
        if len(df) >= n:
            return df.sample(n=n, random_state=42)
        else:
            print(f"Warning: Requested {n} from {len(df)} available. Taking all.")
            return df

    sampled_0 = sample_safe(label_0_df, target_0)
    sampled_1 = sample_safe(label_1_df, target_1)
    
    # If we missed the target count due to shortage, try to fill with the other class
    current_count = len(final_df) + len(sampled_0) + len(sampled_1)
    missing = TARGET_SIZE - current_count
    
    if missing > 0:
        print(f"Still need {missing} samples. Filling from available pool.")
        # Get remaining available
        rem_0 = label_0_df.drop(sampled_0.index)
        rem_1 = label_1_df.drop(sampled_1.index)
        rem_pool = pd.concat([rem_0, rem_1])
        
        if len(rem_pool) >= missing:
            filled = rem_pool.sample(n=missing, random_state=42)
            final_df = pd.concat([final_df, sampled_0, sampled_1, filled])
        else:
            print("Warning: Not enough data to reach 500.")
            final_df = pd.concat([final_df, sampled_0, sampled_1, rem_pool])
    else:
        final_df = pd.concat([final_df, sampled_0, sampled_1])

else:
    print("Overruled samples already exceed 500. Keeping all Overruled samples.")

# Shuffle
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nFinal sampled dataset size: {len(final_df)}")
print("Final label distribution:")
print(final_df['label'].value_counts())

# Display first few rows
display(final_df.head())

Total valid annotated rows (Rating=1): 572
Labels found: [1 2 0]
Overruled (2) count: 43
Label 0 count: 177
Label 1 count: 352
Targeting ~231 of Label 0 and ~226 of Label 1
Still need 54 samples. Filling from available pool.

Final sampled dataset size: 500
Final label distribution:
label
1    280
0    177
2     43
Name: count, dtype: int64


,claim,case_name,type,overruling_case,fact_id,label,facts,question,conclusion,class,Rating,old_negation
0,Foreign organizations outside the United State...,"[""United States Agency for International Devel...",none,NaN,5801,1,The Alliance for Open Society International an...,Does the Court’s decision inU.S. Agency for In...,Because the foreign affiliates of American non...,Supported,1,NaN
1,A guilty plea is valid even without explicit c...,"[""Boykin v. Alabama""]",none,NaN,4861,0,"In the spring of 1966, a series of armed robbe...",Did the trial court commit a reversible error ...,Yes. Justice William O. Douglas delivered the ...,Refuted,1,NaN
2,The death penalty cannot be used for crimes th...,"[""Coker v. Georgia"", ""Herrera v. Collins""]",none,NaN,376,1,"In 1974, Erlich Anthony Coker, serving a numbe...",Was the imposition of the death penalty for th...,"In a 7-to-2 decision, the Court held that the ...",Supported,1,NaN
3,Evidence obtained from an illegal search is in...,"[""Brown v. Illinois"", ""Kaupp v. Texas"", ""New Y...",none,NaN,1395,0,"On May 6, 1968, Roger Corpus was shot and kill...",Should inculpatory statements resulting from a...,No. Justice Harry A. Blackmun delivered the un...,Refuted,1,A person's statement is automatically thrown o...
4,States are not required to assess a person's a...,"[""Turner v. Rogers""]",none,NaN,3785,0,"In January 2007, Michael Turner appeared in Oc...",Do poor people who face incarceration for civi...,No. The Supreme Court reversed and remanded th...,Refuted,1,States are not required to implement procedure...


In [16]:
# Save to CSV
output_file = '../final_test_set.csv'
final_df.to_csv(output_file, index=False)
print(f"Saved final dataset to {output_file}")

Saved final dataset to ../final_test_set.csv


In [17]:
# Clean '*' from claims in both datasets
print("Cleaning '*' from claims...")
# Ensure string type before replacement
df['claim'] = df['claim'].astype(str).str.replace('*', '', regex=False).str.strip()
final_df['claim'] = final_df['claim'].astype(str).str.replace('*', '', regex=False).str.strip()

# Check for duplicate fact_ids in the test set
test_fact_id_counts = final_df['fact_id'].value_counts()
duplicate_fact_ids = test_fact_id_counts[test_fact_id_counts > 1]

if not duplicate_fact_ids.empty:
    print(f"WARNING: Found {len(duplicate_fact_ids)} fact_ids appearing multiple times in the test set:")
    print(duplicate_fact_ids)
else:
    print("No duplicate fact_ids found in the test set.")

Cleaning '*' from claims...
No duplicate fact_ids found in the test set.


In [18]:
import json
import numpy as np

# --- 1. Remove test set claims from the main dataset ---
print(f"Main dataset size before removal: {len(df)}")

# Identify fact_ids to remove (those present in the test set)
test_ids = set(final_df['fact_id'])
df_clean = df[~df['fact_id'].isin(test_ids)].copy()
print(f"Main dataset size after removing test set claims: {len(df_clean)}")

# --- 2. Filter by case_name length > 5 ---
def get_case_name_len(x):
    try:
        val = json.loads(x)
        if isinstance(val, list):
            return len(val)
        return 0 
    except:
        return 0

df_clean['case_name_len'] = df_clean['case_name'].apply(get_case_name_len)
len_before_filter = len(df_clean)
df_clean = df_clean[df_clean['case_name_len'] <= 5].copy()
print(f"Filtered by case_name length > 5. Dropped {len_before_filter - len(df_clean)} rows.")
df_clean.drop(columns=['case_name_len'], inplace=True)

# --- 3. Update label and create class column ---
def get_class(row):
    if row['type'] == 'overruled':
        return 'Overruled'
    # Check if is_negation exists
    if 'is_negation' in row:
        if row['is_negation']:
            return 'Refuted'
        else:
            return 'Supported'
    return 'Supported' # Default fallback

# Ensure is_negation is boolean if it exists
if 'is_negation' in df_clean.columns:
    df_clean['is_negation'] = df_clean['is_negation'].astype(bool)

df_clean['class'] = df_clean.apply(get_class, axis=1)
label_map = {'Refuted': 0, 'Supported': 1, 'Overruled': 2}
df_clean['label'] = df_clean['class'].map(label_map)

# --- 4. Remove duplicate fact_ids (keep random) ---
# Shuffle first to ensure 'first' is random
df_clean = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)
before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['fact_id'], keep='first')
print(f"Dropped {before_dedup - len(df_clean)} duplicate fact_ids from main dataset.")

# --- 5. Drop columns from Big Dataset ---
cols_to_drop_big = ['judgement', 'explanation', 'raw_output', "clean_case_name", 'type', 'fact_id', 'is_negation']
df_clean = df_clean.drop(columns=cols_to_drop_big, errors='ignore')

# --- 6. Drop columns from Small Dataset (Annotated) ---
cols_to_drop_small = ['type', 'fact_id', 'facts', 'question', 'conclusion', 'Rating', 'old_negation']
final_df = final_df.drop(columns=cols_to_drop_small, errors='ignore')

print("Processing complete.")

Main dataset size before removal: 12310
Main dataset size after removing test set claims: 11345
Filtered by case_name length > 5. Dropped 232 rows.
Dropped 5328 duplicate fact_ids from main dataset.
Processing complete.


In [19]:
print("--- Big Dataset (Un-annotated) Head ---")
display(df_clean.head())
print(f"Shape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")

print("\n--- Small Dataset (Annotated) Head ---")
display(final_df.head())
print(f"Shape: {final_df.shape}")
print(f"Columns: {list(final_df.columns)}")

--- Big Dataset (Un-annotated) Head ---


,claim,case_name,overruling_case,label,class
0,"In capital cases involving interracial crimes,...","[""Turner v. Murray""]",NaN,0,Refuted
1,Once a prisoner has earned early release credi...,"[""Lynce v. Mathis""]",NaN,1,Supported
2,ERISA does not prevent states from enacting re...,"[""Kentucky Association of Health Plans, Inc. v...",NaN,0,Refuted
3,Contractual or state law claims for attorney's...,"[""Travelers Casualty & Surety Co. of America v...",NaN,0,Refuted
4,The law requires prisoners to exhaust all admi...,"[""Booth v. Churner""]",NaN,1,Supported


Shape: (5785, 5)
Columns: ['claim', 'case_name', 'overruling_case', 'label', 'class']

--- Small Dataset (Annotated) Head ---


,claim,case_name,overruling_case,label,class
0,Foreign organizations outside the United State...,"[""United States Agency for International Devel...",NaN,1,Supported
1,A guilty plea is valid even without explicit c...,"[""Boykin v. Alabama""]",NaN,0,Refuted
2,The death penalty cannot be used for crimes th...,"[""Coker v. Georgia"", ""Herrera v. Collins""]",NaN,1,Supported
3,Evidence obtained from an illegal search is in...,"[""Brown v. Illinois"", ""Kaupp v. Texas"", ""New Y...",NaN,0,Refuted
4,States are not required to assess a person's a...,"[""Turner v. Rogers""]",NaN,0,Refuted


Shape: (500, 5)
Columns: ['claim', 'case_name', 'overruling_case', 'label', 'class']


In [20]:
# Save Big Dataset
clean_output_file = '../final_dataset_cleaned.csv'
df_clean.to_csv(clean_output_file, index=False)
print(f"Saved cleaned big dataset to {clean_output_file}")

# Save Small Dataset
test_output_file = '../final_test_set_post_clean.csv'
final_df.to_csv(test_output_file, index=False)
print(f"Saved cleaned small dataset to {test_output_file}")

Saved cleaned big dataset to ../final_dataset_cleaned.csv
Saved cleaned small dataset to ../final_test_set_post_clean.csv


In [21]:
import pandas as pd
import json
# --- Identify Cases with Claims ---
final_df = pd.read_csv('../final_test_set_post_clean.csv')
df_clean = pd.read_csv('../final_dataset_cleaned.csv')

# Helper to extract case name from JSON string
def extract_case_name(x):
    try:
        val = json.loads(x)
        if isinstance(val, list) and len(val) > 0:
            return val[0]
        return str(x)
    except:
        return str(x)

# 1. Get cases from annotated set (final_df)
annotated_cases = set(final_df['case_name'].apply(extract_case_name))
print(f"Unique cases in annotated set: {len(annotated_cases)}")

# 2. Get cases from big dataset (df_clean)
big_dataset_cases = set(df_clean['case_name'].apply(extract_case_name))
print(f"Unique cases in big dataset: {len(big_dataset_cases)}")

# 3. Cases in annotated set but NOT in big dataset
cases_only_in_annotated = annotated_cases - big_dataset_cases
print(f"\nCases in annotated set but NOT in big dataset: {len(cases_only_in_annotated)}")
if len(cases_only_in_annotated) > 0:
    print(list(cases_only_in_annotated)[:10]) # Print first 10

# Save these cases
pd.DataFrame({'case_name': list(cases_only_in_annotated)}).to_csv('../cases_only_in_annotated.csv', index=False)
print("Saved cases only in annotated set to ../cases_only_in_annotated.csv")


# --- Identify Cases WITHOUT Claims ---

# Load clean_data_with_details.csv
details_path = '../clean_data_with_details.csv'
try:
    df_details = pd.read_csv(details_path)
    print(f"\nLoaded {details_path}. Shape: {df_details.shape}")
    
    # Get all available cases
    all_cases = set(df_details['name']) # Assuming 'name' column holds the case name
    print(f"Total cases in details file: {len(all_cases)}")
    
    # Combine cases from both datasets that HAVE claims
    cases_with_claims = annotated_cases.union(big_dataset_cases)
    print(f"Total cases with claims (annotated + big): {len(cases_with_claims)}")
    
    # Find cases without claims
    cases_without_claims = all_cases - cases_with_claims
    print(f"\nCases WITHOUT claims: {len(cases_without_claims)}")
    if len(cases_without_claims) > 0:
        print(list(cases_without_claims)[:10]) # Print first 10
        
    # Save these cases
    pd.DataFrame({'case_name': list(cases_without_claims)}).to_csv('../cases_without_claims.csv', index=False)
    print("Saved cases without claims to ../cases_without_claims.csv")
    
except FileNotFoundError:
    print(f"File not found: {details_path}")
except Exception as e:
    print(f"Error processing details file: {e}")

Unique cases in annotated set: 464
Unique cases in big dataset: 2909

Cases in annotated set but NOT in big dataset: 9
['Adams v. Robertson', 'Carachuri-Rosendo v. Holder', 'Pennsylvania v. Mimms', 'McCullen v. Coakley', 'Antonelli v. United States', 'Aguilar v. Felton', 'South Central Bell Telephone Company v. Alabama', 'Berghuis v. Thompkins', 'Ayotte v. Planned Parenthood of Northern New England']
Saved cases only in annotated set to ../cases_only_in_annotated.csv

Loaded ../clean_data_with_details.csv. Shape: (3303, 18)
Total cases in details file: 3240
Total cases with claims (annotated + big): 2918

Cases WITHOUT claims: 322
['Carroll v. United States', 'Sutton v. United Air Lines, Inc.', 'Prunty v. Brooks', 'Cullen v. Pinholster', 'Nitro-Lift Technologies, LLC v. Howard', 'Melendez-Diaz v. Massachusetts', 'Zedner v. United States', 'Wisconsin v. Yoder', 'Meacham v. Knolls Atomic Power Laboratory', 'Cooper Industries, Inc. v. Aviall Services, Inc.']
Saved cases without claims to 

In [22]:
import pandas as pd
import json
import numpy as np

# --- Configuration ---
TEST_SET_PATH = '../final_test_set_post_clean.csv'
BIG_SET_PATH = '../final_dataset_cleaned.csv'
MISSING_CASES_PATH = '../cases_only_in_annotated.csv'
DIFF_CLAIMS_PATH = '../diff_case_claims_with_negatives.csv'
RAW_CLAIMS_PATH = '../claims_raw.csv'

# --- Load Data ---
print("Loading datasets...")
df_test = pd.read_csv(TEST_SET_PATH)
df_big = pd.read_csv(BIG_SET_PATH)
df_cases_missing = pd.read_csv(MISSING_CASES_PATH)
missing_cases_list = df_cases_missing['case_name'].tolist()

print(f"Processing {len(missing_cases_list)} missing cases: {missing_cases_list}")

try:
    df_diff = pd.read_csv(DIFF_CLAIMS_PATH)
    print(f"Loaded diff claims: {df_diff.shape}")
except Exception as e:
    print(f"Error loading diff claims: {e}")
    df_diff = pd.DataFrame()

try:
    df_raw = pd.read_csv(RAW_CLAIMS_PATH)
    print(f"Loaded raw claims: {df_raw.shape}")
except Exception as e:
    print(f"Error loading raw claims: {e}")
    df_raw = pd.DataFrame()

# --- Helpers ---
def parse_case_name(x):
    try:
        val = json.loads(x)
        if isinstance(val, list) and len(val) > 0:
            return val[0]
        return str(x)
    except:
        return str(x)

def clean_claim(x):
    return str(x).replace('*', '').strip()

# Prepare df_diff
if not df_diff.empty:
    df_diff['parsed_case'] = df_diff['case_name'].apply(parse_case_name)

# --- Main Logic ---
new_rows = []

for case in missing_cases_list:
    print(f"\nFinding claims for case: {case}")
    
    # 1. Identify existing claims for this case in the test set (to avoid them)
    existing_claims = set()
    for idx, row in df_test.iterrows():
        row_case = parse_case_name(row['case_name'])
        if row_case == case:
            existing_claims.add(clean_claim(row['claim']))
            
    print(f"  Existing claims in test set ({len(existing_claims)}): {list(existing_claims)}")
    
    found_claim_row = None
    source = None
    
    # 2. Search in diff_case_claims_with_negatives.csv
    if not df_diff.empty:
        # Filter for this case
        candidates = df_diff[df_diff['parsed_case'] == case]
        
        # Filter out negations/contradictions - we want Supported/Consistent
        if 'label' in candidates.columns:
            candidates = candidates[candidates['label'] == 1]
        elif 'judgement' in candidates.columns:
            candidates = candidates[candidates['judgement'] == 'consistent']
            
        # Also check is_negation if it exists
        if 'is_negation' in candidates.columns:
             # Assuming False/NaN is what we want
             candidates = candidates[candidates['is_negation'] != True]

        for idx, row in candidates.iterrows():
            cand_claim = clean_claim(row['claim'])
            if cand_claim not in existing_claims:
                found_claim_row = row
                source = 'diff'
                break
    
    # 3. Search in claims_raw.csv if not found
    if found_claim_row is None and not df_raw.empty:
        candidates = df_raw[df_raw['name'] == case]
        for idx, row in candidates.iterrows():
            cand_claim = clean_claim(row['claim'])
            if cand_claim not in existing_claims:
                found_claim_row = row
                source = 'raw'
                break
                
    # 4. Add to list
    if found_claim_row is not None:
        claim_text = clean_claim(found_claim_row['claim'])
        print(f"  Found new claim from {source}: {claim_text[:50]}...")
        
        new_row = {}
        new_row['claim'] = claim_text
        new_row['case_name'] = json.dumps([case])
        
        # Determine label/class
        if source == 'diff':
            if 'label' in found_claim_row:
                lbl = found_claim_row['label']
            elif 'judgement' in found_claim_row:
                lbl = 1 if found_claim_row['judgement'] == 'consistent' else 0
            else:
                lbl = 1 # Default
                
            new_row['label'] = lbl
            if lbl == 2:
                new_row['class'] = 'Overruled'
            elif lbl == 0:
                new_row['class'] = 'Refuted'
            else:
                new_row['class'] = 'Supported'
                
        else: # raw
            # Assume Supported for raw claims
            new_row['label'] = 1
            new_row['class'] = 'Supported'
            
        new_rows.append(new_row)
    else:
        print(f"  WARNING: No new claim found for {case}")

# --- Save to Big Dataset ---
if new_rows:
    df_new = pd.DataFrame(new_rows)
    
    # Align columns with big dataset
    for col in df_big.columns:
        if col not in df_new.columns:
            df_new[col] = np.nan
            
    # Only keep columns that are in big dataset
    df_new = df_new[df_big.columns]
    
    df_final_big = pd.concat([df_big, df_new], ignore_index=True)
    df_final_big.to_csv(BIG_SET_PATH, index=False)
    print(f"\nAdded {len(new_rows)} new claims to big dataset. Saved to {BIG_SET_PATH}")
    print(f"New big dataset shape: {df_final_big.shape}")
else:
    print("\nNo new claims added.")

Loading datasets...
Processing 9 missing cases: ['Adams v. Robertson', 'Carachuri-Rosendo v. Holder', 'Pennsylvania v. Mimms', 'McCullen v. Coakley', 'Antonelli v. United States', 'Aguilar v. Felton', 'South Central Bell Telephone Company v. Alabama', 'Berghuis v. Thompkins', 'Ayotte v. Planned Parenthood of Northern New England']
Loaded diff claims: (11818, 10)
Loaded raw claims: (10471, 3)

Finding claims for case: Adams v. Robertson
  Existing claims in test set (1): ['Federal courts will not consider constitutional issues that state courts did not address in their rulings.']
  Found new claim from diff: When a state court issues a brief, unexplained dec...

Finding claims for case: Carachuri-Rosendo v. Holder
  Existing claims in test set (1): ['Immigration authorities cannot classify a state drug conviction as an aggravated felony based on prior convictions that were not part of the original state court case.']
  Found new claim from raw: A simple drug possession conviction under 

In [24]:
# Load the human-annotated subset
df_anno = pd.read_csv("../final_test_set_post_clean.csv")

print(f"Annotated Dataset Shape: {df_anno.shape}")
print(f"\nColumn Names:\n{list(df_anno.columns)}")
print(f"\nFirst few rows:")
display(df_anno.head())

print(f"\n--- Basic Statistics ---")
print(f"Total annotated rows: {len(df_anno)}")

# Check for annotation columns
annotation_cols = [col for col in df_anno.columns if col not in df.columns]
print(f"\nAnnotation-specific columns: {annotation_cols}")

# If there's a Rating or similar column, show distribution
if 'Rating' in df_anno.columns:
    print(f"\n--- Rating Distribution ---")
    print(df_anno['Rating'].value_counts())
    print(f"\nRatings (normalized):")
    print(df_anno['Rating'].value_counts(normalize=True))
    
    # Check how many are annotated vs not
    annotated_count = df_anno['Rating'].notna().sum()
    print(f"\nAnnotated: {annotated_count}")
    print(f"Not yet annotated: {len(df_anno) - annotated_count}")

# Check if fact_ids match the main dataset
if 'fact_id' in df_anno.columns:
    matching_ids = df_anno['fact_id'].isin(df['fact_id']).sum()
    print(f"\nFact IDs matching main dataset: {matching_ids}/{len(df_anno)}")

print(f"\n--- Label Distribution in Annotated Set ---")
if 'label' in df_anno.columns:
    print(df_anno['label'].value_counts())

print(f"\n--- Missing Values ---")
print(df_anno.isnull().sum())

Annotated Dataset Shape: (500, 5)

Column Names:
['claim', 'case_name', 'overruling_case', 'label', 'class']

First few rows:


,claim,case_name,overruling_case,label,class
0,Foreign organizations outside the United State...,"[""United States Agency for International Devel...",NaN,1,Supported
1,A guilty plea is valid even without explicit c...,"[""Boykin v. Alabama""]",NaN,0,Refuted
2,The death penalty cannot be used for crimes th...,"[""Coker v. Georgia"", ""Herrera v. Collins""]",NaN,1,Supported
3,Evidence obtained from an illegal search is in...,"[""Brown v. Illinois"", ""Kaupp v. Texas"", ""New Y...",NaN,0,Refuted
4,States are not required to assess a person's a...,"[""Turner v. Rogers""]",NaN,0,Refuted



--- Basic Statistics ---
Total annotated rows: 500

Annotation-specific columns: ['class']

--- Label Distribution in Annotated Set ---
label
1    280
0    177
2     43
Name: count, dtype: int64

--- Missing Values ---
claim                0
case_name            0
overruling_case    457
label                0
class                0
dtype: int64


In [25]:
import pandas as pd

# --- Filter clean_data_with_details.csv ---
DETAILS_PATH = '../clean_data_with_details.csv'
NO_CLAIMS_PATH = '../cases_without_claims.csv'
OUTPUT_PATH = '../supreme_court_cases.csv'

print(f"Loading {DETAILS_PATH}...")
df_details = pd.read_csv(DETAILS_PATH)
print(f"Original shape: {df_details.shape}")

try:
    # Load the list of cases to drop
    df_no_claims = pd.read_csv(NO_CLAIMS_PATH)
    cases_to_drop = set(df_no_claims['case_name'])
    print(f"Found {len(cases_to_drop)} cases to drop (no claims).")
    
    # Identify the case name column in the details file
    case_col = 'name'
    if case_col not in df_details.columns:
        if 'case_name' in df_details.columns:
            case_col = 'case_name'
        else:
            print(f"Warning: 'name' column not found. Columns are: {list(df_details.columns)}")
            # Stop if we can't find the column
            raise ValueError("Could not identify case name column.")

    # Filter
    df_filtered = df_details[~df_details[case_col].isin(cases_to_drop)].copy()
    
    print(f"Filtered shape: {df_filtered.shape}")
    print(f"Dropped {len(df_details) - len(df_filtered)} rows.")

    # Save
    df_filtered.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved filtered cases to {OUTPUT_PATH}")

except Exception as e:
    print(f"Error processing files: {e}")

Loading ../clean_data_with_details.csv...
Original shape: (3303, 18)
Found 322 cases to drop (no claims).
Filtered shape: (2978, 18)
Dropped 325 rows.
Saved filtered cases to ../supreme_court_cases.csv
